In [1]:
import re
import unicodedata

def load_abbreviation_dict(file_path):
    abbr_dict = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip().lower()
            if not line or ',' not in line:
                continue
            abbr, full = line.split(',', 1)
            abbr_dict[abbr.strip()] = full.strip()
    return abbr_dict

def load_synonym_dict(file_path):
    syn_dict = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip().lower()
            if not line or ',' not in line:
                continue
            syns, canonical = line.split(',', 1)
            syn_tuple = tuple(s.strip() for s in syns.split('|'))
            syn_dict[syn_tuple] = canonical.strip()
    return syn_dict

def load_keywords(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip().lower() for line in f if line.strip()]

def clean_text(text):
    text = re.sub(r'[\+\-\•\–\—]', ',', text)
    text = re.sub(r'(Page|Trang) ?\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'(NHÀ XUẤT BẢN|MỤC LỤC|Đáp án|Số đăng ký|Mã số|Chủ biên|Tham gia biên soạn|Lời giới thiệu|Đáp án)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\.{5,}', '', text)
    text = re.sub(r'\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\(Hình\s?\d+\)', '', text)
    text = re.sub(r'\n{2,}', '\n', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C')
    text = text.lower()
    return text.strip()

def standardize_terms(text, abbreviation_dict, synonym_dict):
    for abbr, full in abbreviation_dict.items():
        text = re.sub(r'\b{}\b'.format(re.escape(abbr)), full, text)
    for syns, canonical in synonym_dict.items():
        for syn in syns:
            text = re.sub(r'\b{}\b'.format(re.escape(syn)), canonical, text)
    return text

def sentence_split(text, keywords):
    sentences = re.split(r'(?<=[\.?!])\s+|\n', text)
    filtered = []
    keywords_lower = [kw.lower() for kw in keywords]
    for sent in sentences:
        sent = sent.strip()
        if len(sent) > 10 and not re.match(
            r'^(bài|chương|mục tiêu|hình|ts\.|ths\.|pgs\.|gs\.|số đăng ký|mã số|trình bày|chủ biên|tham gia biên soạn|lời giới thiệu|đáp án|phần|mục|số trang|trang|page|\.{3,})',
            sent, re.IGNORECASE):
            if any(kw in sent for kw in keywords_lower):
                filtered.append(sent)
    return filtered

def preprocess_file(file_path, abbreviation_dict, synonym_dict, keywords):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
    cleaned = clean_text(raw_text)
    standardized = standardize_terms(cleaned, abbreviation_dict, synonym_dict)
    sentences = sentence_split(standardized, keywords)
    return sentences

if __name__ == "__main__":
    file_input = "benh_hoc_quanh_rang_dai_hoc_y_ha_noi.txt"
    abbr_file = "abbreviations.txt"
    syn_file = "synonyms.txt"
    keyword_file = "keywords.txt"

    abbreviation_dict = load_abbreviation_dict(abbr_file)
    synonym_dict = load_synonym_dict(syn_file)
    keywords = load_keywords(keyword_file)

    sentences = preprocess_file(file_input, abbreviation_dict, synonym_dict, keywords)
    with open("processed_sentences.txt", "w", encoding="utf-8") as f:
        for sent in sentences:
            f.write(sent + "\n")
